In [ ]:
import json
import os
import random
import re
import shutil
import traceback
from pathlib import Path
from typing import TypedDict


class Instance(TypedDict):
    bbox: tuple[float, float, float, float] | None
    cls_idx: int | None
    keypoints: list[tuple[float, float, int] | None]


def normalize_coords(x: int, y: int, w: int, h: int) -> tuple[float, float]:
    """归一化坐标"""
    return x / w, y / h


def get_name(idx: int, label: str, desc: str) -> str:
    if desc:
        return desc.split("_")[0] if "_" in desc else desc
    elif label in keypoint_map:
        # 只有self有关键点
        return "self"
    elif label == "self":
        return "self"
    else:
        return f"{label}{idx}"


def labelme2yolo(
    json_path: str,
) -> str:
    """将labelme格式的标注文件转换为yolo格式"""
    with open(json_path, "r") as f:
        data = json.load(f)
    img_path = data["imagePath"]
    img_w = data["imageWidth"]
    img_h = data["imageHeight"]

    instances = {}
    for i, shape in enumerate(data["shapes"]):
        label = shape["label"]
        points = shape["points"]
        desc = shape["description"] or ""

        name = get_name(idx=i, label=label, desc=desc)
        if not name:
            print(
                f"Skipping {name} from description: {desc}, shape: {shape}, json: {json_path}"
            )
        if name not in instances:
            instances[name] = Instance(
                bbox=None, cls_idx=None, keypoints=[None] * len(keypoint_map)
            )

        # 处理矩形框
        if shape["shape_type"] == "rectangle" and label in class_map:
            x_coords = [p[0] for p in points]
            y_coords = [p[1] for p in points]
            x_min, x_max = min(x_coords), max(x_coords)
            y_min, y_max = min(y_coords), max(y_coords)

            x_center = (x_min + x_max) / 2
            y_center = (y_min + y_max) / 2
            shp_w = x_max - x_min
            shp_h = y_max - y_min

            norm_x_center, norm_y_center = normalize_coords(
                x_center, y_center, img_w, img_h
            )
            norm_w, norm_h = normalize_coords(shp_w, shp_h, img_w, img_h)
            instances[name]["bbox"] = (norm_x_center, norm_y_center, norm_w, norm_h)
            instances[name]["cls_idx"] = class_map[label]

        # 处理关键点
        elif shape["shape_type"] == "point" and label in keypoint_map:
            kp_index = keypoint_map[label]
            x, y = points[0]
            norm_x, norm_y = normalize_coords(x, y, img_w, img_h)
            instances[name]["keypoints"][kp_index] = (norm_x, norm_y, 2)

    # 由instances生成yolo格式文本
    yolo_lines = []
    for name, inst in instances.items():
        if inst["bbox"] is None or inst["cls_idx"] is None:
            print(f"Skipping instance: {name} missing bbox or cls_idx from {json_path}")
            continue

        line = [str(inst["cls_idx"])]
        line.extend([f"{coord:6f}" for coord in inst["bbox"]])
        # 顺序补充关键点
        for kp_idx in keypoint_map.values():
            kp = inst["keypoints"][kp_idx]
            if kp is None:
                # 关键点标记为不存在
                line.extend(["0.0", "0.0", "0"])
            else:
                # 关键点标记为可见
                line.extend([f"{kp[0]:6f}", f"{kp[1]:6f}", str(kp[2])])

        yolo_lines.append(" ".join(line))

    # 保存
    filename = os.path.splitext(img_path)[0] + ".txt"
    output = os.path.join(os.path.dirname(json_path), filename)
    with open(output, "w") as f:
        f.write("\n".join(yolo_lines))
    print(f"Successfully converted {json_path} to {output}")

    return output


def make_dataset(labelme_folder: str, dataset_root: str, val_ratio=0.2, seed=None):
    if seed is not None:
        random.seed(seed)
    image_files = [
        f
        for f in os.listdir(labelme_folder)
        if f.lower().endswith(
            (
                ".jpg",
                ".png",
                ".jpeg",
            )
        )
    ]
    json_files = [f for f in os.listdir(labelme_folder) if f.lower().endswith(".json")]
    image_names = {Path(f).stem for f in image_files}
    json_names = {Path(f).stem for f in json_files}

    if image_names != json_names:
        missing_jsons = image_names - json_names
        missing_images = json_names - image_names
        if missing_jsons:
            print(f"Missing JSON files: {missing_jsons}")
        if missing_images:
            print(f"Missing image files: {missing_images}")
        common_names = image_names & json_names
    else:
        common_names = image_names

    all_names = list(common_names)
    random.shuffle(all_names)
    num_val = int(len(all_names) * val_ratio)
    val_names = set(all_names[:num_val])
    train_names = set(all_names[num_val:])

    print(f"Total images: {len(all_names)}")
    print(f"Train images: {len(train_names)}")
    print(f"Val images: {len(val_names)}")

    split_dirs = {
        "train_images": os.path.join(dataset_root, "images", "train"),
        "train_labels": os.path.join(dataset_root, "labels", "train"),
        "val_images": os.path.join(dataset_root, "images", "val"),
        "val_labels": os.path.join(dataset_root, "labels", "val"),
    }
    for dir_path in split_dirs.values():
        os.makedirs(dir_path, exist_ok=True)

    for name in all_names:
        src_img = os.path.join(
            labelme_folder, next(f for f in image_files if Path(f).stem == name)
        )
        src_json = os.path.join(
            labelme_folder, next(f for f in json_files if Path(f).stem == name)
        )
        src_lbl = labelme2yolo(src_json)

        if name in train_names:
            dst_img = os.path.join(
                split_dirs["train_images"], os.path.basename(src_img)
            )
            dst_lbl = os.path.join(
                split_dirs["train_labels"], os.path.basename(src_lbl)
            )
        elif name in val_names:
            dst_img = os.path.join(split_dirs["val_images"], os.path.basename(src_img))
            dst_lbl = os.path.join(split_dirs["val_labels"], os.path.basename(src_lbl))

        shutil.copy2(src_img, dst_img)
        shutil.copy2(src_lbl, dst_lbl)

    print(f"Done. Copied {len(all_names)} files.")


# ================================= 配置和运行 =================================
label_folders = [
    "./dataset/images_minimap1",
    "./dataset/images_minimap2",
    "./dataset/images_minimap3",
]
label_folder = "./dataset/images_minimap"
dataset_root = "./dataset/dataset_minimap"
val_ratio = 0.1
random_seed = 42

class_map = {"self": 0, "ally": 1, "enemy": 2}
keypoint_map = {"bow": 0, "stern": 1, "port": 2, "stbd": 3}

yaml_name = "data_minimap.yaml"

yaml_content = f"""# {yaml_name}
train: {os.path.join(os.path.abspath(dataset_root), "images", "train")}
val: {os.path.join(os.path.abspath(dataset_root), "images", "val")}

nc: 3

names: ['self', 'ally', 'enemy', ]

kpt_shape: [4, 3]  # self的4个关键点, 每个关键点有3个坐标
labels: ['bow', 'stern', 'port', 'stbd']
flip_idx: []
"""

if os.path.exists(dataset_root):
    shutil.rmtree(dataset_root)
    os.mkdir(dataset_root)

if os.path.exists(label_folder):
    shutil.rmtree(label_folder)
    os.mkdir(label_folder)
for folder in label_folders:
    if not os.path.exists(folder):
        continue
    shutil.copytree(folder, label_folder, dirs_exist_ok=True)

try:
    make_dataset(label_folder, dataset_root, val_ratio, random_seed)
    yaml_path = os.path.join(Path(dataset_root).parent, yaml_name)
    with open(yaml_path, "w", encoding="utf-8") as yaml_file:
        yaml_file.write(yaml_content)
    print(f"Successfully created dataset")
except Exception as e:
    traceback.print_exc()

shutil.rmtree(label_folder)

In [ ]:
import torch

# 检查 CUDA 是否可用
print("CUDA 可用:", torch.cuda.is_available())

# 检查 PyTorch 是否能识别到您的 GPU
print("GPU 数量:", torch.cuda.device_count())

# 打印 GPU 名称
if torch.cuda.is_available():
    print("GPU 名称:", torch.cuda.get_device_name(0))

# 检查 PyTorch 使用的 CUDA 版本
print("PyTorch CUDA 版本:", torch.version.cuda)

In [ ]:
level = "11s"  # n, s, m, l, x
model_name = f"minimap_{level}"
print(model_name)

In [ ]:
import os
import shutil

import gc
import torch
from ultralytics import YOLO

if os.path.exists(f"runs/pose/{model_name}"):
    shutil.rmtree(f"runs/pose/{model_name}")

# --- 1. 加载模型 ---
# 加载预训练的Pose模型（推荐用于迁移学习）
model = YOLO(f"yolo{level}-pose.pt")
data = r"./dataset/data_minimap.yaml"

# --- 2. 训练模型 ---
# 使用您准备好的 data 文件进行训练
try:
    results = model.train(
    data=data,                  # 指向您的数据配置文件
    epochs=100,                 # 训练轮数
    time=20.0,                  # 训练时长（小时）
    patience=300,               # 早停阈值 连续300个epoch没有提升则停止训练
    imgsz=640,                  # 输入图像大小
    batch=-1,                   # 批次大小 (根据GPU显存调整)
    name=f'{model_name}',       # 本次训练的名称
    workers=2,
    # half=True,                  # 半精度训练
    augment=True,               # 启用增强（默认开启）
    # hsv_h=0.015,                # 色调抖动
    # hsv_s=0.7,                  # 饱和度抖动（增强颜色鲁棒性）
    # hsv_v=0.4,                  # 亮度抖动
    # degrees=10,                 # 旋转
    # translate=0.1,              # 平移
    # scale=0.5,                  # 缩放
    # shear=2.0,                  # 剪切
    # flipud=0.0,                 # 上下翻转
    # fliplr=0.5,                 # 左右翻转
    # mosaic=1.0,                 # Mosaic 增强（4图拼接，提升小目标检测）
    # mixup=0.1,                  # MixUp：混合两张图（对抗过拟合）
    # copy_paste=0.2,             # Copy-Paste：将目标粘贴到新位置（模拟密集场景）
)
except Exception as e:
    print(e)
finally:
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
from ultralytics import YOLO
model = YOLO(f"./runs/pose/{model_name}/weights/best.pt")

res = model.export(format="onnx")
print(res)

In [ ]:
import os
import random
import sys
import cv2
from collections import defaultdict
from glob import glob
import numpy as np
from ultralytics import YOLO

imgpaths = glob("./dataset/images_warship1/*.[pP][nN][gG]")
if not imgpaths:
    print("没有找到任何png或jpg格式的图片")
    sys.exit(1)

imgpath = random.choice(imgpaths)
img = cv2.imread(imgpath)
if img is None:
    print("图片加载失败")
    sys.exit(1)
    
h, w = img.shape[:2]
roi = img[h-350:h, w-350:w]

# 推理
model = YOLO(f"./runs/pose/{model_name}/weights/best.pt")

def read_minimap(model_minimap: YOLO, roi: np.ndarray) -> dict[str, list[np.ndarray]] | None:
    """
    Returns: {"self": [arr(x, y), arr(dx, dy)],
                "ally": [arr(x1, y1), arr(x2, y2),...],
                "enemy": [arr(x1, y1), arr(x2, y2),...]}
    """
    if model_minimap is None:
        return None

    # predict
    data = defaultdict(list)
    self_conf = 0.0
    self_center = None
    self_kps = None
    result = model_minimap.predict(roi, conf=0.5, iou=0.5)[0]
    if result.boxes is None or result.keypoints is None:
        return None
    
    cls_ids = result.boxes.cls.cpu().numpy()
    confs =  result.boxes.conf.cpu().numpy()
    xywhs = result.boxes.xywh.cpu().numpy()
    kpts = result.keypoints.xy.cpu().numpy()

    for i in range(len(cls_ids)):
        label = result.names[int(cls_ids[i])]
        conf = confs[i]
        center = xywhs[i][:2]
        data[label].append(center)
        if label == "self" and conf > self_conf:
            self_conf = conf
            self_center = center
            self_kps = kpts[i]
    if self_kps is None:
        return None
    
    # handle keypoints of self
    print(f"Self keypoints has {len(self_kps)} {self_kps}")
    bow, stern, port, stbd = [p for p in self_kps[:4]]
    points = np.array([bow, self_center, stern, port, stbd])
    a, b = np.polyfit(points[:, 0], points[:, 1], 1)
    delta = np.array([1.0, a])
    data["self"] = [self_center, delta]
    
    show = True
    if show:
        frame = result.plot()
        cv2.imshow("minimap", frame)
        cv2.waitKey(3000)
        cv2.destroyAllWindows()

    return data

data = read_minimap(model, roi)
print(data)

In [ ]:

# result = {"self": [(x, y), (dx, dy)],
#           "ally": [(x1, y1), (x2, y2),...],
#           "enemy": [(x1, y1), (x2, y2),...]}

from dataclasses import dataclass

@dataclass
class MapData:
    self: list[tuple[int, int]]
    ally: list[tuple[int, int]]
    enemy: list[tuple[int, int]]
    
    
class MapReader:
    
    def __init__(self, map_data: MapData):
        self.read_data(map_data)
        
    def read_data(self, map_data: MapData):
        point_self, delta = map_data.self
        self.point_self = np.array(point_self)
        self.polar_axis = np.array(delta) - self.point_self
        # polar-axis angle for xy-axis
        self.polar_angle = np.arctan2(self.polar_axis[1], self.polar_axis[0])
        
    def convert_points(self, xy_points: list[tuple[int, int]]) -> list[tuple[float, float]]:
        points = np.array(xy_points) - self.point_self
        dists = np.linalg.norm(points, axis=1)
        point_angles = np.arctan2(points[:, 1], points[:, 0])  # angle for xy-axis
        angle_diffs = point_angles - self.polar_angle
        angles = np.degrees(angle_diffs % (2 * np.pi))
        
        return [(float(d), float(a)) for d, a in zip(dists, angles)]

# test
result = {'self': [(1, 1), (2, 1)],
          'enemy':[(2,2), (3, 4), (5, 6)],
          'ally': [(7, 8), (9, 10)],}
map_data = MapData(**result)
reader = MapReader(map_data)
enemies = result['enemy']
res = reader.convert_points(enemies)
print(res)
[(1.4142135623730951, 45.0), (3.605551275463989, 56.309932474020215), (6.4031242374328485, 51.34019174590991)]